### Phase 4: Experiments 5 & 6 (Layer Freezing & Knowledge Distillation)

This complete, standalone notebook contains the setup, dataset processing, and the final two experiments required to complete Phase 4. It maintains your `SAMPLES_PER_EPOCH = 12000` limit and uses the same 1024-token head+tail truncation strategy.

*   **Experiment 5 (Layer Freezing):** GPT-2 with layers 0–8 frozen, fine-tuning only the top 3 layers and the classification head.
*   **Experiment 6 (Knowledge Distillation):** Distills knowledge from your best Phase 4 model (Prefix-LM + Adapters) into a 6-layer Prefix-LM student.


### Cell 1: Setup & Initialization


In [ ]:
import os, random, json, time, math, zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient
from collections import Counter
from safetensors.torch import load_file

# --- GPU & Env Setup ---
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF2")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

# --- Configuration ---
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
MODEL_PATHS = {
    "gpt2":     "/kaggle/input/datasets/himangshumahato/model1",
    "prefixlm": "/kaggle/input/datasets/himangshumahato/model4",
}
# IMPORTANT: Point this to your saved Experiment 4 checkpoint
TEACHER_CHECKPOINT = "/kaggle/working/cs781_phase4/04_prefixlm_adapters/best_model.pt"

MODEL_CONTEXT_LENGTH = 1024
N_EMBD, N_LAYER, N_HEAD, N_KV_HEAD = 768, 12, 12, 4
HEAD_TOKENS, TAIL_TOKENS = 512, 256
DOC_MAX_LEN = HEAD_TOKENS + TAIL_TOKENS
SAMPLES_PER_EPOCH = 12000

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

BASE_OUTPUT_DIR = "/kaggle/working/cs781_phase4_part2"
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

def experiment_dirs(name):
    d = os.path.join(BASE_OUTPUT_DIR, name)
    os.makedirs(d, exist_ok=True)
    return {
        "dir": d,
        "model_path": os.path.join(d, "best_model.pt"),
        "run_info_path": os.path.join(d, "run_info.json"),
        "prediction_path": os.path.join(d, "prediction.json"),
        "submission_zip": os.path.join(d, "submission.zip"),
    }


### Cell 2: Tokenizer Loading
*(Ensure `a1_tokenizer.py` is in your working directory from the previous run, or paste the ByteLevelBPE class code here if running in a completely clean environment).*


In [ ]:
%%writefile a1_tokenizer.py
"""From-scratch byte-level BPE tokenizer (GPT-2 style), no external tokenizer libs.

Differences from the plain-bytes version:
  * Text is first split into "words" with a GPT-2 style regex, and merges never
    cross word boundaries. Training and encoding then work on *unique words*
    (with counts) instead of the whole corpus, which is orders of magnitude faster.
  * Training uses incremental pair counts + a lazy max-heap (no full rescan per merge).
  * Encoding applies merges per word by rank, with a word -> ids cache.
  * One special token, <|endoftext|>, used as EOS / PAD / BOS.

Interface:
    tok.encode(text, add_eos=False) -> list[int]
    tok.decode(ids, skip_special_tokens=True) -> str
    tok.vocab_size / tok.get_vocab_size()
    tok.eos_token_id, tok.pad_token_id, tok.bos_token_id
    tok.save(path) / ByteLevelBPE.load(path)
"""

from __future__ import annotations

import heapq
import json
import time
from collections import Counter, defaultdict

import regex as re  # `pip install regex` (needed for \p{L} / \p{N})

# GPT-2 pre-tokenization pattern
_PAT = re.compile(
    r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
)

EOS_TOKEN = "<|endoftext|>"


class ByteLevelBPE:
    def __init__(self):
        # ids 0-255 = raw bytes; 256.. = learned merges; last id = <|endoftext|>
        self.merges: dict[tuple[int, int], int] = {}      # (a, b) -> new_id (id order == rank order)
        self.itos: dict[int, bytes] = {i: bytes([i]) for i in range(256)}
        self.eos_token_id = 256
        self.vocab_size = 257
        self._cache: dict[str, tuple] = {}

    # alias attributes so the tokenizer plugs into HF-style code
    @property
    def pad_token_id(self): return self.eos_token_id
    @property
    def bos_token_id(self): return self.eos_token_id

    def __len__(self): return self.vocab_size
    def get_vocab_size(self): return self.vocab_size

    def train(self, text: str, vocab_size: int, verbose: bool = False) -> None:
        """Learn (vocab_size - 257) merges; the final id is reserved for <|endoftext|>."""
        if vocab_size < 257:
            raise ValueError("vocab_size must be at least 257 (256 bytes + EOS)")
        num_merges = vocab_size - 257
        t0 = time.time()

        # 1) pre-tokenize and count unique words
        word_counts = Counter(_PAT.findall(text))
        words = [list(w.encode("utf-8")) for w in word_counts]
        freqs = list(word_counts.values())
        if verbose:
            print(f"[bpe] {len(words):,} unique words from {sum(freqs):,} total words")

        # 2) initial pair statistics
        stats: dict[tuple[int, int], int] = defaultdict(int)
        where: dict[tuple[int, int], set] = defaultdict(set)
        for wi, w in enumerate(words):
            f = freqs[wi]
            for p in zip(w, w[1:]):
                stats[p] += f
                where[p].add(wi)

        # lazy max-heap: (-count, a, b) -> highest count first, ties -> smallest (a, b)
        heap = [(-c, a, b) for (a, b), c in stats.items()]
        heapq.heapify(heap)

        self.merges, self.itos = {}, {i: bytes([i]) for i in range(256)}

        # 3) merge loop
        for i in range(num_merges):
            pair, count = None, 0
            while heap:
                negc, a, b = heapq.heappop(heap)
                if stats.get((a, b), 0) == -negc:   # skip stale heap entries
                    pair, count = (a, b), -negc
                    break
            if pair is None or count < 2:
                break

            new_id = 256 + i
            self.merges[pair] = new_id
            self.itos[new_id] = self.itos[pair[0]] + self.itos[pair[1]]

            touched = set()
            for wi in list(where[pair]):
                w, f = words[wi], freqs[wi]
                # remove this word's old pair contributions
                for p in zip(w, w[1:]):
                    stats[p] -= f
                    where[p].discard(wi)
                    touched.add(p)
                # apply merge (leftmost, non-overlapping)
                a, b = pair
                out, j, n = [], 0, len(w)
                while j < n:
                    if j < n - 1 and w[j] == a and w[j + 1] == b:
                        out.append(new_id); j += 2
                    else:
                        out.append(w[j]); j += 1
                words[wi] = out
                # add new pair contributions
                for p in zip(out, out[1:]):
                    stats[p] += f
                    where[p].add(wi)
                    touched.add(p)

            for p in touched:
                c = stats.get(p, 0)
                if c > 0:
                    heapq.heappush(heap, (-c, p[0], p[1]))
                else:
                    stats.pop(p, None)
                    where.pop(p, None)

            if verbose and (i % 1000 == 0 or i == num_merges - 1):
                print(f"[bpe] merge {i + 1}/{num_merges} | pair={pair} count={count:,} "
                      f"| elapsed={time.time() - t0:.1f}s")

        self.eos_token_id = 256 + len(self.merges)
        self.vocab_size = self.eos_token_id + 1
        self._cache = {}

    # Inference
    def _encode_word(self, word: str) -> tuple:
        hit = self._cache.get(word)
        if hit is not None:
            return hit
        ids = list(word.encode("utf-8"))
        merges = self.merges
        while len(ids) > 1:
            # pick the applicable merge learned earliest (lowest id)
            best, best_id = None, None
            for p in zip(ids, ids[1:]):
                nid = merges.get(p)
                if nid is not None and (best_id is None or nid < best_id):
                    best, best_id = p, nid
            if best is None:
                break
            a, b = best
            out, j, n = [], 0, len(ids)
            while j < n:
                if j < n - 1 and ids[j] == a and ids[j + 1] == b:
                    out.append(best_id); j += 2
                else:
                    out.append(ids[j]); j += 1
            ids = out
        res = tuple(ids)
        if len(self._cache) > 2_000_000:
            self._cache.clear()
        self._cache[word] = res
        return res

    def encode(self, text: str, add_eos: bool = False) -> list[int]:
        ids: list[int] = []
        for w in _PAT.findall(text):
            ids.extend(self._encode_word(w))
        if add_eos:
            ids.append(self.eos_token_id)
        return ids

    def decode(self, ids, skip_special_tokens: bool = True) -> str:
        if hasattr(ids, "tolist"):
            ids = ids.tolist()
        parts = []
        for i in ids:
            if i == self.eos_token_id:
                if not skip_special_tokens:
                    parts.append(EOS_TOKEN.encode("utf-8"))
                continue
            parts.append(self.itos[i])
        return b"".join(parts).decode("utf-8", errors="replace")

    def save(self, path: str) -> None:
        data = {
            "merges": [[a, b, nid] for (a, b), nid in self.merges.items()],
            "vocab_size": self.vocab_size,
            "eos_token_id": self.eos_token_id,
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f)

    @classmethod
    def load(cls, path: str) -> "ByteLevelBPE":
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls()
        for a, b, nid in data["merges"]:
            tok.merges[(a, b)] = nid
            tok.itos[nid] = tok.itos[a] + tok.itos[b]
        tok.eos_token_id = data["eos_token_id"]
        tok.vocab_size = data["vocab_size"]
        return tok


def train_tokenizer(train_text: str, vocab_size: int):
    """Train a byte-level BPE tokenizer on `train_text` and return it."""
    tok = ByteLevelBPE()
    tok.train(train_text, vocab_size, verbose=True)
    return tok

In [ ]:
from a1_tokenizer import ByteLevelBPE

tokenizer = ByteLevelBPE.load(os.path.join(MODEL_PATHS["gpt2"], "bpe_tokenizer.json"))
PAD_ID = tokenizer.eos_token_id
print("Vocabulary size:", len(tokenizer))


### Cell 3: Data Loading & Processing


In [ ]:
def hf_json(path):
    p = hf_hub_download(repo_id=HF_DATASET_REPO, filename=path, repo_type="dataset", token=HF_TOKEN)
    with open(p, encoding="utf-8") as f:
        return json.load(f)

train_raw = hf_json("Phase_4_CJPE/Train/train.json")
dev_raw   = hf_json("Phase_4_CJPE/Train/dev.json")
test_raw  = hf_json("Phase_4_CJPE/Test/test.json")

def to_records(raw, with_labels=True):
    LABEL_KEYS = ["label", "pred", "judgment", "outcome", "y"]
    items = raw.items() if isinstance(raw, dict) else [(d.get("id", d.get("case_id")), d) for d in raw]
    records, label_key = [], None
    for doc_id, d in items:
        text = d if isinstance(d, str) else d.get("text", d.get("case_text", d.get("document")))
        rec = {"id": doc_id, "text": text}
        if with_labels:
            if label_key is None:
                label_key = next(k for k in LABEL_KEYS if k in d)
            rec["label"] = int(d.get(label_key))
        records.append(rec)
    return records

train_records = to_records(train_raw, with_labels=True)
dev_records   = to_records(dev_raw,   with_labels=True)
test_records  = to_records(test_raw,  with_labels=False)

rng = np.random.RandomState(42)
dev_records_small = [dev_records[i] for i in rng.permutation(len(dev_records))[:2500]]

train_label_counts = Counter(r["label"] for r in train_records)
total_labels = sum(train_label_counts.values())
class_weights = torch.tensor([total_labels / (2 * train_label_counts[l]) for l in (0, 1)], dtype=torch.float).to(device)

def encode_head_tail(text, head=HEAD_TOKENS, tail=TAIL_TOKENS):
    ids = tokenizer.encode(text)
    return ids if len(ids) <= head + tail else ids[:head] + ids[-tail:]

class CJPEDataset(torch.utils.data.Dataset):
    def __init__(self, records, with_labels=True):
        self.records = records
        self.with_labels = with_labels
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        r = self.records[idx]
        if "ids" not in r:
            r["ids"] = np.array(encode_head_tail(r["text"]), dtype=np.int32)
        item = {"ids": r["ids"], "idx": idx}
        if self.with_labels: item["label"] = r["label"]
        return item

def cjpe_collator(batch):
    B, L = len(batch), DOC_MAX_LEN
    input_ids = torch.full((B, L), PAD_ID, dtype=torch.long)
    attention_mask = torch.zeros((B, L), dtype=torch.bool)
    for i, item in enumerate(batch):
        n = len(item["ids"])
        input_ids[i, :n] = torch.tensor(item["ids"])
        attention_mask[i, :n] = True
    out = {"input_ids": input_ids, "attention_mask": attention_mask, "record_idx": torch.tensor([i["idx"] for i in batch])}
    if "label" in batch[0]: out["labels"] = torch.tensor([i["label"] for i in batch])
    return out

def make_train_loader(batch_size):
    ds = CJPEDataset(train_records)
    sampler = torch.utils.data.RandomSampler(ds, replacement=False, num_samples=min(SAMPLES_PER_EPOCH, len(ds)))
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, sampler=sampler, collate_fn=cjpe_collator)

def make_dev_loader(batch_size=8):
    return torch.utils.data.DataLoader(CJPEDataset(dev_records_small), batch_size=batch_size, shuffle=False, collate_fn=cjpe_collator)


### Cell 4: Architecture Definitions


In [ ]:
# Required Architectures for GPT-2 (Layer Freezing) and Prefix-LM (Knowledge Distillation)
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_positions, dropout=0.1):
        super().__init__()
        self.n_head, self.n_embd, self.head_dim = n_head, n_embd, n_embd // n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout, self.resid_dropout = nn.Dropout(dropout), nn.Dropout(dropout)
        self.register_buffer("bias", torch.tril(torch.ones(n_positions, n_positions)).view(1, 1, n_positions, n_positions))

    def forward(self, x, layer_past=None, attn_mask=None):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        seq_len = T + (layer_past[0].size(-2) if layer_past is not None else 0)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        if attn_mask is not None:
            att = att.masked_fill(~attn_mask, float('-inf'))
        else:
            att = att.masked_fill(self.bias[:, :, :T, :seq_len] == 0, float('-inf'))
        y = (self.attn_dropout(F.softmax(att, dim=-1)) @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), (k, v)

class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd); self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * n_embd, n_embd); self.dropout = nn.Dropout(dropout)
    def forward(self, x): return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_positions, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd); self.attn = CausalSelfAttention(n_embd, n_head, n_positions, dropout)
        self.ln_2 = nn.LayerNorm(n_embd); self.mlp = MLP(n_embd, dropout)
    def forward(self, x, layer_past=None, attn_mask=None):
        a, present = self.attn(self.ln_1(x), layer_past=layer_past, attn_mask=attn_mask)
        x = x + a; x = x + self.mlp(self.ln_2(x))
        return x, present

class GPT2ScratchLM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([TransformerBlock(n_embd, n_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)

# Prefix-LM Dependencies
class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, n_positions, dropout=0.1):
        super().__init__()
        self.n_head, self.n_kv_head = n_head, n_kv_head
        self.head_dim, self.num_groups = n_embd // n_head, n_head // n_kv_head
        self.q_proj = nn.Linear(n_embd, n_embd)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout, self.resid_dropout = nn.Dropout(dropout), nn.Dropout(dropout)
        self.register_buffer("bias", torch.tril(torch.ones(n_positions, n_positions)).view(1, 1, n_positions, n_positions))

    def forward(self, x, layer_past=None, attn_mask=None):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        past_len = layer_past[0].size(-2) if layer_past is not None else 0
        seq_len = T + past_len
        k = k.repeat_interleave(self.num_groups, dim=1); v = v.repeat_interleave(self.num_groups, dim=1)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        if attn_mask is not None: att = att.masked_fill(~attn_mask, float('-inf'))
        else: att = att.masked_fill(self.bias[:, :, past_len:past_len + T, :seq_len] == 0, float('-inf'))
        y = (self.attn_dropout(F.softmax(att, dim=-1)) @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), (k, v)

class GQATransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, n_positions, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, n_positions, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4*n_embd), nn.GELU(approximate='tanh'), nn.Linear(4*n_embd, n_embd), nn.Dropout(dropout))
    def forward(self, x, layer_past=None, attn_mask=None):
        a, present = self.attn(self.ln_1(x), layer_past=layer_past, attn_mask=attn_mask)
        x = x + a; x = x + self.mlp(self.ln_2(x))
        return x, present

class PrefixLM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, n_kv_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([GQATransformerBlock(n_embd, n_head, n_kv_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)

class DocClassifier(nn.Module):
    def __init__(self, lm, n_embd, dropout=0.1):
        super().__init__()
        self.lm = lm
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(2 * n_embd, n_embd), nn.GELU(approximate='tanh'),
            nn.Dropout(dropout), nn.Linear(n_embd, 2)
        )

    def forward(self, input_ids, attention_mask=None, labels=None, class_weights=None, threshold_offset=None):
        B, T = input_ids.shape
        x = self.lm.drop(self.lm.wte(input_ids) + self.lm.wpe(torch.arange(T, device=input_ids.device)))
        causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=input_ids.device)).view(1, 1, T, T)
        attn_mask = causal & attention_mask[:, None, None, :].bool() if attention_mask is not None else causal

        for block in self.lm.h:
            x, _ = block(x, attn_mask=attn_mask)
        x = self.lm.ln_f(x)

        last_idx = attention_mask.sum(dim=1) - 1 if attention_mask is not None else torch.full((B,), T - 1, device=input_ids.device)
        m = attention_mask.unsqueeze(-1).float() if attention_mask is not None else torch.ones(B, T, 1, device=input_ids.device)
        last = x[torch.arange(B, device=input_ids.device), last_idx]
        mean = ((x.float() * m).sum(dim=1) / m.sum(dim=1).clamp(min=1)).to(x.dtype)

        logits = self.head(self.dropout(torch.cat([last, mean], dim=-1)))
        if threshold_offset is not None: logits = logits + threshold_offset
        loss = F.cross_entropy(logits, labels, weight=class_weights) if labels is not None else None
        return {"loss": loss, "logits": logits}


### Cell 5: Standard Training & Metrics Functions


In [ ]:
def macro_prf_binary(y_true, y_pred):
    cnts = np.zeros((2, 3))
    for t, p in zip(y_true, y_pred):
        cnts[p, 1] += 1; cnts[t, 2] += 1
        if t == p: cnts[t, 0] += 1
    P = np.divide(cnts[:, 0], cnts[:, 1], out=np.zeros(2), where=cnts[:, 1] > 0)
    R = np.divide(cnts[:, 0], cnts[:, 2], out=np.zeros(2), where=cnts[:, 2] > 0)
    F = np.divide(2 * P * R, P + R, out=np.zeros(2), where=(P + R) > 0)
    return P, R, F

@torch.no_grad()
def get_logits(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    for b in loader:
        b = {k: v.to(device) for k, v in b.items() if k != "record_idx"}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            all_logits.append(model(input_ids=b["input_ids"], attention_mask=b["attention_mask"])["logits"].float().cpu())
        if "labels" in b: all_labels.append(b["labels"].cpu())
    return torch.cat(all_logits), (torch.cat(all_labels) if all_labels else None)

def tune_threshold_offset(logits, labels, n_steps=41):
    best_off, best_f1, y = 0.0, macro_prf_binary(labels.numpy(), logits.argmax(-1).numpy())[2].mean(), labels.numpy()
    for off in np.linspace(-3.0, 3.0, n_steps):
        shifted = logits.clone(); shifted[:, 1] += off
        f1 = macro_prf_binary(y, shifted.argmax(-1).numpy())[2].mean()
        if f1 > best_f1: best_off, best_f1 = off, f1
    return best_off, best_f1

def write_submission(paths, model, offset=0.0):
    test_loader = torch.utils.data.DataLoader(CJPEDataset(test_records, with_labels=False), batch_size=8, shuffle=False, collate_fn=cjpe_collator)
    logits, _ = get_logits(model, test_loader)
    if offset: logits[:, 1] += offset
    preds = logits.argmax(-1).tolist()
    prediction = [{"id": r["id"], "pred": int(p)} for r, p in zip(test_records, preds)]
    with open(paths["prediction_path"], "w", encoding="utf-8") as f: json.dump(prediction, f, ensure_ascii=False)
    with zipfile.ZipFile(paths["submission_zip"], "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(paths["prediction_path"], arcname="prediction.json")
    print(f"Saved: {paths['submission_zip']}")


### Cell 6: Experiment 5 - GPT-2 + Layer Freezing


In [ ]:
EXP5_NAME = "05_gpt2_layer_freeze"
paths = experiment_dirs(EXP5_NAME)

EPOCHS, BATCH_SIZE, GRAD_ACCUM = 15, 8, 2
LR_FROZEN_FT, LR_HEAD, WEIGHT_DECAY = 1e-4, 1e-3, 0.05

# 1. Initialize & Load Backbone
backbone = GPT2ScratchLM(vocab_size=tokenizer.vocab_size, n_positions=MODEL_CONTEXT_LENGTH, n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD)
backbone.load_state_dict(load_file(os.path.join(MODEL_PATHS["gpt2"], "model.safetensors")), strict=False)

# 2. Apply Layer Freezing (Freeze layers 0-8. Unfreeze layers 9-11 and LayerNorm)
for name, param in backbone.named_parameters():
    if any(f"h.{i}." in name for i in range(9, 12)) or "ln_f" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

model = DocClassifier(backbone, n_embd=N_EMBD).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"[{EXP5_NAME}] Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

train_loader = make_train_loader(BATCH_SIZE)
dev_loader = make_dev_loader(8)

optimizer = torch.optim.AdamW([
    {"params": [p for n, p in model.named_parameters() if p.requires_grad and "head" not in n], "lr": LR_FROZEN_FT},
    {"params": [p for n, p in model.named_parameters() if p.requires_grad and "head" in n], "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)

scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
best_f1, bad_epochs, history = 0.0, 0, {"train": [], "dev_f1": []}

# Standard Loop
for epoch in range(EPOCHS):
    model.train(); run, cnt = 0.0, 0
    optimizer.zero_grad()
    for i, b in enumerate(train_loader):
        b = {k: v.to(device) for k, v in b.items() if k != "record_idx"}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            out = model(**b, class_weights=class_weights)
        scaler.scale(out["loss"] / GRAD_ACCUM).backward()
        run += out["loss"].item(); cnt += 1
        if (i + 1) % GRAD_ACCUM == 0 or (i + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad()

    logits, labels = get_logits(model, dev_loader)
    f1 = macro_prf_binary(labels.numpy(), logits.argmax(-1).numpy())[2].mean()
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {run/cnt:.4f} | Dev F1: {f1:.4f}")
    if f1 > best_f1:
        best_f1, bad_epochs = f1, 0
        torch.save(model.state_dict(), paths["model_path"])
    else:
        bad_epochs += 1
        if bad_epochs >= 5: print("Early stopping"); break

model.load_state_dict(torch.load(paths["model_path"], weights_only=True))
logits, labels = get_logits(model, dev_loader)
offset, final_f1 = tune_threshold_offset(logits, labels)

run_info = {
    "experiment": EXP5_NAME, "technique": "Layer Freezing (Top 3)",
    "best_dev_f1": best_f1, "final_dev_f1": final_f1,
    "trainable_params": trainable, "total_params": total, "threshold_offset": offset
}
with open(paths["run_info_path"], "w") as f: json.dump(run_info, f, indent=2)
write_submission(paths, model, offset)


### Cell 7: Experiment 6 - Knowledge Distillation (Prefix-LM)


In [ ]:
EXP6_NAME = "06_prefixlm_distillation"
paths = experiment_dirs(EXP6_NAME)

EPOCHS, BATCH_SIZE, GRAD_ACCUM = 20, 8, 2
LR_STUDENT, KD_ALPHA, KD_TEMP = 3e-4, 0.5, 2.0

# 1. Load Teacher (Full 12-layer Prefix-LM with Adapters)
# Note: You MUST use the Adapter wrapper logic from Exp 4 if the teacher utilized adapters!
class AdapterBottleneck(nn.Module):
    def __init__(self, n_embd, bottleneck=64):
        super().__init__()
        self.down = nn.Linear(n_embd, bottleneck)
        self.act = nn.GELU(approximate='tanh')
        self.up = nn.Linear(bottleneck, n_embd)
    def forward(self, x): return x + self.up(self.act(self.down(x)))

class AdapterBlockWrapper(nn.Module):
    def __init__(self, block, n_embd, bottleneck=64):
        super().__init__()
        self.block, self.adapter = block, AdapterBottleneck(n_embd, bottleneck)
    def forward(self, x, layer_past=None, attn_mask=None):
        x, present = self.block(x, layer_past=layer_past, attn_mask=attn_mask)
        return self.adapter(x), present

teacher_lm = PrefixLM(vocab_size=tokenizer.vocab_size, n_positions=MODEL_CONTEXT_LENGTH, n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD, n_kv_head=N_KV_HEAD)
for i in range(len(teacher_lm.h)): teacher_lm.h[i] = AdapterBlockWrapper(teacher_lm.h[i], N_EMBD, 64)
teacher = DocClassifier(teacher_lm, n_embd=N_EMBD).to(device)

print(f"Loading Teacher from {TEACHER_CHECKPOINT}...")
teacher.load_state_dict(torch.load(TEACHER_CHECKPOINT, weights_only=True, map_location=device))
for p in teacher.parameters(): p.requires_grad = False
teacher.eval()

# 2. Initialize Student (6-Layer Prefix-LM, no adapters)
student_lm = PrefixLM(vocab_size=tokenizer.vocab_size, n_positions=MODEL_CONTEXT_LENGTH, n_embd=N_EMBD, n_layer=6, n_head=N_HEAD, n_kv_head=N_KV_HEAD)
student = DocClassifier(student_lm, n_embd=N_EMBD).to(device)

trainable = sum(p.numel() for p in student.parameters() if p.requires_grad)
print(f"[{EXP6_NAME}] Student Trainable params: {trainable:,}")

train_loader = make_train_loader(BATCH_SIZE)
dev_loader = make_dev_loader(8)

optimizer = torch.optim.AdamW(student.parameters(), lr=LR_STUDENT, weight_decay=0.05)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
best_f1, bad_epochs = 0.0, 0

# KD Loop
for epoch in range(EPOCHS):
    student.train(); run, cnt = 0.0, 0
    optimizer.zero_grad()
    for i, b in enumerate(train_loader):
        b = {k: v.to(device) for k, v in b.items() if k != "record_idx"}

        with torch.no_grad():
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                t_logits = teacher(input_ids=b["input_ids"], attention_mask=b["attention_mask"])["logits"]

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            s_out = student(input_ids=b["input_ids"], attention_mask=b["attention_mask"], labels=b.get("labels"), class_weights=class_weights)
            s_logits = s_out["logits"]

            # KD Loss computation
            loss_ce = s_out["loss"]
            loss_kl = F.kl_div(
                F.log_softmax(s_logits / KD_TEMP, dim=-1),
                F.softmax(t_logits / KD_TEMP, dim=-1),
                reduction="batchmean"
            ) * (KD_TEMP ** 2)
            loss = (KD_ALPHA * loss_ce) + ((1.0 - KD_ALPHA) * loss_kl)

        scaler.scale(loss / GRAD_ACCUM).backward()
        run += loss.item(); cnt += 1

        if (i + 1) % GRAD_ACCUM == 0 or (i + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad()

    logits, labels = get_logits(student, dev_loader)
    f1 = macro_prf_binary(labels.numpy(), logits.argmax(-1).numpy())[2].mean()
    print(f"Epoch {epoch+1}/{EPOCHS} | KD Loss: {run/cnt:.4f} | Dev F1: {f1:.4f}")

    if f1 > best_f1:
        best_f1, bad_epochs = f1, 0
        torch.save(student.state_dict(), paths["model_path"])
    else:
        bad_epochs += 1
        if bad_epochs >= 5: print("Early stopping"); break

student.load_state_dict(torch.load(paths["model_path"], weights_only=True))
logits, labels = get_logits(student, dev_loader)
offset, final_f1 = tune_threshold_offset(logits, labels)

run_info = {
    "experiment": EXP6_NAME, "technique": "Knowledge Distillation",
    "teacher_model": "Prefix-LM (12L + Adapters)", "student_model": "Prefix-LM (6L)",
    "best_dev_f1": best_f1, "final_dev_f1": final_f1,
    "trainable_params": trainable, "threshold_offset": offset
}
with open(paths["run_info_path"], "w") as f: json.dump(run_info, f, indent=2)
write_submission(paths, student, offset)
